In [1]:
# Cell 1: Setup and Imports
import re
import os
import json
import torch
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
from tqdm.auto import tqdm 
from haversine import haversine
import matplotlib.pyplot as plt
import torch.nn.functional as F
from scipy.stats import entropy
warnings.filterwarnings('ignore')
from itertools import permutations
from itertools import combinations
from joblib import Parallel, delayed
from geopy.geocoders import Nominatim
from torch_geometric.nn import GCNConv
from sklearn.model_selection import KFold
from torch_geometric.data import HeteroData
from sklearn.preprocessing import MinMaxScaler
from geopy.extra.rate_limiter import RateLimiter
from sklearn.metrics.pairwise import haversine_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration
DATA_PATH = "Incidents_imputed.xlsx" 
MAX_WORKERS = 4

In [2]:
# Cell 2: Data Loader with Chunking
def load_data():
    return pd.read_excel(
        DATA_PATH,
        parse_dates=['Job OFF Time', 'Job ON Time'],
        engine='openpyxl'
    )

df = load_data()

prediction_window = 180

print(f"Loaded {len(df)} records")

Loaded 292829 records


In [4]:
# Cell 3: Centroid Calculation (Simplified without temporal split)
def calculate_centroids_and_feeders(df, cache_dir="temporal_centroids", city_overrides=None, json_file="city_coords.json"):
    """
    Implementation with caching, validation, and fallback geolocation 
    using the full dataset (i.e. without temporal filtering).
    """
    os.makedirs(cache_dir, exist_ok=True)
    # Use a fixed cache filename since we no longer depend on a cutoff date
    cache_file = os.path.join(cache_dir, "centroids_full.parquet")
    
    # If a cached centroids file exists, load and validate it.
    if os.path.exists(cache_file):
        centroids = pd.read_parquet(cache_file)
        if centroids[['centroid_lat', 'centroid_lon']].isna().any().any():
            print("Cached centroids contain NaN. Recomputing centroids...")
            os.remove(cache_file)
        else:
            return centroids
    
    # Instead of temporal filtering, use the full dataset
    hist_df = df.copy()
    
    # --- Geocoding with Caching ---
    city_cache_file = os.path.join(cache_dir, "city_coords_full.parquet")
    
    if os.path.exists(city_cache_file):
        geocoded = pd.read_parquet(city_cache_file)
    else:
        # Load JSON file for fallback geolocation
        with open(json_file, "r") as file:
            city_coords_raw = json.load(file)
            city_coords = {k.strip().upper(): v for k, v in city_coords_raw.items()}  # Normalize keys
        
        unique_cities = hist_df['Job City_imputed'].str.strip().str.upper().unique()
        geocoder = Nominatim(user_agent="temporal_centroids", timeout=20)
        
        coords = {}
        missing_from_both = []
        for city in tqdm(unique_cities, desc="Geocoding Cities"):
            # Use city overrides if provided
            if city_overrides and city in city_overrides:
                value = city_overrides[city]
                if isinstance(value, list):
                    value = (value[0], value[1])
                coords[city] = value
                continue
                
            try:
                location = geocoder.geocode(f"{city}, Oklahoma")
                if location:
                    coords[city] = (location.latitude, location.longitude)
                else:
                    raise ValueError("Geolocation failed")
            except:
                # Attempt fallback from JSON (case-insensitive)
                fallback_coords = city_coords.get(city.strip().upper())
                if fallback_coords:
                    if isinstance(fallback_coords, list):
                        coords[city] = (fallback_coords[0], fallback_coords[1])
                    else:
                        coords[city] = fallback_coords
                else:
                    coords[city] = (None, None)
                    missing_from_both.append(city)
        
        geocoded = pd.DataFrame({
            'Job City_imputed': list(coords.keys()),
            'lat': [c[0] for c in coords.values()],
            'lon': [c[1] for c in coords.values()]
        }).drop_duplicates()
        
        # Ensure numeric coordinates
        geocoded['lat'] = pd.to_numeric(geocoded['lat'], errors='coerce')
        geocoded['lon'] = pd.to_numeric(geocoded['lon'], errors='coerce')
        
        geocoded.to_parquet(city_cache_file)
        
        if missing_from_both:
            print(f"Warning: The following cities were not found in either geocoding or the JSON file: {missing_from_both}")
    
    # Validate geocoding results
    missing_geo = geocoded[geocoded['lat'].isna() | geocoded['lon'].isna()]
    if not missing_geo.empty:
        print(f"Final missing coordinates for cities: {missing_geo['Job City_imputed'].tolist()}")
    
    # --- Weighted Centroid Calculation ---
    city_weights = (
        hist_df.groupby(['Job Substation', 'Job City_imputed'])
        .size()
        .reset_index(name='incident_count')
    )
    
    # Normalize city names for merging
    city_weights['Job City_normalized'] = city_weights['Job City_imputed'].str.strip().str.upper()
    
    # Merge with geocoded coordinates
    merged = city_weights.merge(
        geocoded,
        left_on='Job City_normalized',
        right_on='Job City_imputed',
        how='left'
    )
    
    # Drop rows with invalid coordinates
    merged = merged.dropna(subset=['lat', 'lon'])
    
    if (merged['incident_count'] <= 0).any():
        raise ValueError("Invalid incident counts (≤0) detected after merging.")
    
    # Calculate centroids
    centroids = (
        merged.groupby('Job Substation')
        .apply(lambda g: pd.Series({
            'centroid_lat': np.average(g['lat'], weights=g['incident_count']),
            'centroid_lon': np.average(g['lon'], weights=g['incident_count']),
            'cities': g['Job City_imputed_x'].tolist()  # original city names from city_weights
        })).reset_index()
    )
    
    if centroids[['centroid_lat', 'centroid_lon']].isna().any().any():
        missing_subs = centroids[centroids.isna().any(axis=1)]
        raise ValueError(f"Substations with missing coordinates: {missing_subs['Job Substation'].tolist()}")
    
    # --- Feeder Aggregation ---
    feeder_agg = (
        hist_df.groupby('Job Substation')['Feeder ID']
        .agg(lambda x: list(set(x)) if not x.empty else [])
        .reset_index(name='feeders')
    )
    
    centroids = centroids.merge(feeder_agg, on='Job Substation', how='left')
    
    # Note: Temporal metadata has been removed
    centroids.to_parquet(cache_file)
    
    return centroids

# Example usage:
# Now we call the function using the full dataset (here, 'df' is the full data)
centroids = calculate_centroids_and_feeders(df)


In [5]:
# Cell 4: Spatial Layer

def build_spatial_layer(centroids):
    # Shared cities calculation (with symmetric matrix)
    substation_cities = centroids.set_index('Job Substation')['cities']
    city_matrix = pd.DataFrame(
        index=substation_cities.index,
        columns=substation_cities.index,
        dtype=float
    )
    for (a, b) in combinations(substation_cities.index, 2):
        shared = len(set(substation_cities[a]) & set(substation_cities[b]))
        city_matrix.loc[a, b] = shared
        city_matrix.loc[b, a] = shared  # Mirror to lower triangle

    # Shared feeders calculation (with symmetric matrix)
    substation_feeders = centroids.set_index('Job Substation')['feeders']
    feeder_matrix = pd.DataFrame(
        index=substation_feeders.index,
        columns=substation_feeders.index,
        dtype=float
    )
    for (a, b) in combinations(substation_feeders.index, 2):
        shared = len(set(substation_feeders[a]) & set(substation_feeders[b]))
        feeder_matrix.loc[a, b] = shared
        feeder_matrix.loc[b, a] = shared  # Mirror to lower triangle

    # Distance calculation (Haversine)
    coords = centroids[['centroid_lat', 'centroid_lon']].values
    distances = haversine_distances(np.radians(coords)) * 6371000  # in meters

    # Build edges based solely on distance (≤ 20km)
    spatial_edges = []
    for i in range(len(coords)):
        for j in range(i+1, len(coords)):
            sub_i = centroids.iloc[i]['Job Substation']
            sub_j = centroids.iloc[j]['Job Substation']
            distance = distances[i, j]
            
            if distance <= 20000:  # ONLY DISTANCE CONDITION
                shared_cities = city_matrix.loc[sub_i, sub_j]
                shared_feeders = feeder_matrix.loc[sub_i, sub_j]
                spatial_edges.append({
                    'source': sub_i,
                    'target': sub_j,
                    'distance': distance,
                    'shared_cities': shared_cities,
                    'shared_feeders': shared_feeders
                })
    
    spatial_df = pd.DataFrame(spatial_edges)
    
    return spatial_df

# Example usage:
spatial_edges = build_spatial_layer(centroids)
num_edges = len(spatial_edges)
print(f"Number of edges created: {num_edges}")


Number of edges created: 6162


In [6]:
# Cell 5 : Temporal Layer 

def build_temporal_layer(df, time_cutoff=None):
    """Build temporal edges from events occurring BEFORE a cutoff time"""
    if time_cutoff is not None:
        df = df[df['Job OFF Time'] < time_cutoff].copy()
        
    temporal = df.sort_values('Job OFF Time').copy()
    
    # Existing code for edge creation
    temporal['next_substation'] = temporal['Job Substation'].shift(-1)
    temporal['next_off'] = temporal['Job OFF Time'].shift(-1)
    temporal['time_diff'] = (temporal['next_off'] - temporal['Job OFF Time']).dt.total_seconds() / 60
    
    edges = temporal[(temporal['time_diff'] <= 60) & temporal['time_diff'].notna()].copy()
    
    # Aggregation remains unchanged
    edges = edges.groupby(['Job Substation', 'next_substation']).agg(
        min_duration=('time_diff', 'min'),
        event_count=('time_diff', 'count')
    ).reset_index().rename(columns={
        'Job Substation': 'source',
        'next_substation': 'target'
    })
    
    valid_subs = set(df['Job Substation'])
    return edges[edges['target'].isin(valid_subs)]

# Revised usage: Using the full dataset without a time cutoff
temporal_edges = build_temporal_layer(df, time_cutoff=None)
print(f"Temporal edges created using the full dataset: {len(temporal_edges)}")


Temporal edges created using the full dataset: 59908


In [7]:
# Cell 6: Causal Layer

def build_causal_layer(df, centroids, spatial_edges, max_distance=20000, n_jobs=-1):

    # 1. Validate inputs against the full dataset
    subs = set(df['Job Substation'].unique())
    centroid_subs = set(centroids['Job Substation'])
    if not centroid_subs.issubset(subs):
        raise ValueError("Centroids contain substations not present in the full dataset")
    
    # 2. Prepare the full dataset for causal analysis (use all available events)
    causal_df = df.copy()
    causal_df = causal_df[['Job Substation', 'Cause Desc', 'Job OFF Time']]
    causal_df['Job OFF Time'] = pd.to_datetime(causal_df['Job OFF Time'], errors='coerce')
    causal_df = causal_df.dropna(subset=['Job OFF Time'])
    
    # 3. Initialize spatial neighbors from the full dataset
    spatial_pairs = set()
    for _, row in spatial_edges.iterrows():
        spatial_pairs.add(frozenset({row['source'], row['target']}))
    
    # 4. Safe coordinate mapping (precompute substation coordinates)
    coord_map = centroids.set_index('Job Substation')[['centroid_lat', 'centroid_lon']].to_dict('index')
    
    # 5. Failure timeline preparation (group by substation and cause)
    failure_times = (
        causal_df.groupby(['Job Substation', 'Cause Desc'])['Job OFF Time']
        .apply(lambda x: np.sort(x.values))
        .reset_index()
        .rename(columns={'Job OFF Time': 'failure_times'})
    )
    
    # 6. Leak-proof processing core: process each cause group
    def process_cause(cause, group):
        edges = []
        valid_subs = group['Job Substation'].unique()
        
        for a, b in permutations(valid_subs, 2):
            # Spatial relationship check: ensure a and b are neighbors
            if frozenset({a, b}) not in spatial_pairs:
                continue
                
            # Retrieve failure sequences for substations a and b
            a_times = group[group['Job Substation'] == a]['failure_times'].iloc[0]
            b_times = group[group['Job Substation'] == b]['failure_times'].iloc[0]
            
            # Enforce temporal causality: last failure at a must occur before first failure at b
            if a_times[-1] >= b_times[0]:
                continue
                
            # Calculate time differences (a causes b)
            time_diffs = b_times[:, None] - a_times[None, :]
            valid_pairs = time_diffs > np.timedelta64(0, 'ns')
            if not np.any(valid_pairs):
                continue
            
            # Convert mean time difference to hours
            mean_diff = np.mean(time_diffs[valid_pairs])
            mean_diff_seconds = mean_diff / np.timedelta64(1, 's')
            mean_lag = mean_diff_seconds / 3600
            
            # Spatial validation: calculate distance between substations a and b
            distance = haversine(
                (coord_map[a]['centroid_lat'], coord_map[a]['centroid_lon']),
                (coord_map[b]['centroid_lat'], coord_map[b]['centroid_lon'])
            ) * 1000  # convert to meters
            
            if distance <= max_distance:
                edges.append({
                    'source': a,
                    'target': b,
                    'cause': cause,  # use the cause key from the grouping
                    'mean_lag_hrs': mean_lag,
                    'distance_m': distance,
                    'event_count': len(a_times) + len(b_times)
                })
        return edges

    # 7. Parallel execution over causes (using events from the full dataset)
    from joblib import Parallel, delayed  # ensure joblib is imported
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_cause)(cause, group) 
        for cause, group in failure_times.groupby('Cause Desc')
    )
    
    # 8. Normalize causal edge features (training-only normalization replaced by full-dataset normalization)
    causal_edges = pd.DataFrame([e for sublist in results for e in sublist])
    if not causal_edges.empty:
        from sklearn.preprocessing import StandardScaler, MinMaxScaler
        time_scaler = StandardScaler().fit(causal_edges[['mean_lag_hrs']])
        dist_scaler = MinMaxScaler().fit(causal_edges[['distance_m']])
        
        causal_edges['norm_lag'] = time_scaler.transform(causal_edges[['mean_lag_hrs']])
        causal_edges['norm_dist'] = dist_scaler.transform(causal_edges[['distance_m']])
        
        # Apply threshold on normalized distance (if max_distance is 20000, threshold=1)
        causal_edges = causal_edges[causal_edges['norm_dist'] <= (max_distance / 20000)]
    else:
        causal_edges = pd.DataFrame(columns=[
            'source', 'target', 'cause', 'mean_lag_hrs',
            'distance_m', 'event_count', 'norm_lag', 'norm_dist'
        ])
        time_scaler, dist_scaler = None, None

    # Final validation: ensure edges contain only substations present in the full dataset
    invalid_sources = causal_edges[~causal_edges['source'].isin(subs)]
    invalid_targets = causal_edges[~causal_edges['target'].isin(subs)]
    if not invalid_sources.empty or not invalid_targets.empty:
        raise RuntimeError("Causal edges contain substations not present in the dataset")
    
    print(f"Built causal layer with {len(causal_edges)} edges")
    return causal_edges, time_scaler, dist_scaler

# Usage with the full dataset (no fixed cutoff)
causal_edges, time_scaler, dist_scaler = build_causal_layer(
    df=df,
    centroids=centroids,
    spatial_edges=spatial_edges,
    n_jobs=8
)


Built causal layer with 5632 edges


In [8]:
# Cell 7: Node Features Without Target Leakage 
def create_node_features(df, centroids, scaler=None):
    
    # 1. Use direct reference (no copy needed)
    hist_df = df
    
    # 2. Base feature engineering (safe columns only)
    node_features = hist_df.groupby('Job Substation').agg(
        incident_count=('Job OFF Time', 'count')
    )
    
    # 3. Temporal recency features
    reference_date = hist_df['Job OFF Time'].max()
    last_incident = hist_df.groupby('Job Substation')['Job OFF Time'].max()
    node_features['days_since_last_incident'] = (reference_date - last_incident).dt.days
    
    # 4. Geo features from centroids (use only existing columns)
    geo_features = centroids.set_index('Job Substation')[[
        'centroid_lat', 'centroid_lon'
    ]].copy()
    
    node_features = node_features.merge(
        geo_features, 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    # 5. Define numerical columns (no categoricals or missing features)
    numerical_cols = [
        'incident_count', 
        'days_since_last_incident',
        'centroid_lat', 
        'centroid_lon'
    ]
    
    # 6. Ensure numerical columns exist
    for col in numerical_cols:
        if col not in node_features:
            node_features[col] = 0
    
    # 7. Normalization
    from sklearn.preprocessing import StandardScaler
    if scaler is None:
        scaler = StandardScaler().fit(node_features[numerical_cols])
    
    node_features[numerical_cols] = scaler.transform(node_features[numerical_cols])
    
    return node_features[numerical_cols], scaler

# Revised usage: Create node features for the full dataset (no pre-split)
node_features_full, feature_scaler = create_node_features(
    df,        # Use the full dataset
    centroids,
    scaler=None  # Scaler is fit on the full dataset
)

print("Node features created for the full dataset:")
print(node_features_full.head())


Node features created for the full dataset:
                    incident_count  days_since_last_incident  centroid_lat  \
Job Substation                                                               
3109:HONOR HEIGHTS        0.031886                 -0.398078      0.137795   
3110:RIVERSIDE            0.575400                 -0.407828      0.135893   
3111:FIVE TRIBES          0.313535                 -0.405391      0.135922   
3114: TENNYSON           -0.895695                  0.525769      0.135893   
3114:TENNYSON             2.017398                 -0.405391      0.135710   

                    centroid_lon  
Job Substation                    
3109:HONOR HEIGHTS      0.440649  
3110:RIVERSIDE          0.444501  
3111:FIVE TRIBES        0.444269  
3114: TENNYSON          0.444501  
3114:TENNYSON           0.444725  


In [9]:
# Cell 8: Target Variable with 180-Day Window
def create_target_variable(df, severe_causes=None, critical_equipment=None, return_thresholds=False):
    """
    Creates target labels using the full dataset with a 180-day window.
    
    Steps:
      1. Compute thresholds and automatically detect severe causes and critical equipment.
      2. Sort the data by Job Substation, Equip Desc, and Job OFF Time.
      3. For each (Job Substation, Equip Desc) group, compute days until the next incident.
      4. Flag an incident as needing replacement if the next incident occurs after 180 days 
         (or if no subsequent incident exists).
      5. Aggregate the flag at the substation level: if any incident for the substation is flagged, 
         the substation is marked as needing replacement.
    
    Parameters:
      - df: Full dataset DataFrame.
      - severe_causes: (Optional) Pre-defined severe causes.
      - critical_equipment: (Optional) Pre-defined critical equipment.
      - return_thresholds: If True, also return the computed severe causes and critical equipment.
    
    Returns:
      - final_targets: A Series indexed by Job Substation with binary labels (1 = needs replacement, 0 = does not).
    """
    # 1. Calculate cause thresholds using the full dataset
    cause_stats = df.groupby('Cause Desc').agg(
        median_duration=('Job Duration Mins', 'median'),
        max_customers=('Custs Affected', 'max')
    )
    cause_thresholds = cause_stats.quantile(0.75)
    
    # 2. Identify severe causes from the full dataset
    if severe_causes is None:
        severe_causes = cause_stats[
        (cause_stats['median_duration'] > cause_thresholds['median_duration']) &
        (cause_stats['max_customers'] > cause_thresholds['max_customers'])
    ].index.unique()

    
    # 3. Identify critical equipment from the full dataset
    equip_stats = df.groupby('Equip Desc').agg(
    failure_freq=('Job Display ID', 'count'),  # Use Job Display ID for consistency
    saidi=('Job SAIDI', 'mean')
    )

    equipment_thresholds = equip_stats.quantile(0.75)
    
    if critical_equipment is None:
        critical_equipment = df.groupby('Equip Desc').filter(
            lambda x: (len(x) > equipment_thresholds['failure_freq']) and
                      (x['Job SAIDI'].mean() > equipment_thresholds['saidi'])
        )['Equip Desc'].unique()
    
    # 4. Sort data and calculate days until next incident per (substation, equipment)
    df_sorted = df.sort_values(['Job Substation', 'Equip Desc', 'Job OFF Time']).copy()
    df_sorted['days_until_next_incident'] = df_sorted.groupby(
        ['Job Substation', 'Equip Desc']
    )['Job OFF Time'].diff(-1).dt.days.abs()
    
    # 5. Filter to incidents with severe causes and critical equipment
    filtered_incidents = df_sorted[
        df_sorted['Cause Desc'].isin(severe_causes) &
        df_sorted['Equip Desc'].isin(critical_equipment)
    ].copy()
    
    # 6. Flag replacement: if the next incident is more than 180 days away or absent, flag as needing replacement
    filtered_incidents['needs_replacement'] = np.where(
        (filtered_incidents['days_until_next_incident'] > 180) | 
        (filtered_incidents['days_until_next_incident'].isna()),
        1,
        0
    )
    
    # 7. Aggregate at the substation level:
    # If any (substation, equipment) group indicates a need for replacement, mark the substation as 1.
    final_targets = filtered_incidents.groupby('Job Substation')['needs_replacement'].max()
    
    if return_thresholds:
        return final_targets, severe_causes, critical_equipment
    else:
        return final_targets

# Usage: Now using the full dataset with a 180-day observation window for target creation
full_targets, full_severe_causes, full_critical_equipment = create_target_variable(
    df, return_thresholds=True
)


In [10]:
# Cell 9: Safe Heterogeneous Graph Construction
def build_heterogeneous_graph(node_features, target_variable, edge_dataframes):
    # 1. Strict node validation
    valid_subs = node_features.index.tolist()
    data = HeteroData()
    
    # 2. Node features validation
    if node_features.isna().any().any():
        raise ValueError("Node features contain missing values")
        
    # 3. Convert node features to tensor with explicit type conversion
    data['substation'].x = torch.tensor(
        node_features.values.astype(np.float32),
        dtype=torch.float32
    )
    
    # 4. Validate and convert target variable:
    if not target_variable.index.equals(node_features.index):
        print("Warning: Target variable index does not match node features.")
        print("Reindexing target variable to match node features and filling missing targets with 0.")
        target_variable = target_variable.reindex(node_features.index, fill_value=0)

    data['substation'].y = torch.tensor(
        target_variable.values.astype(np.float32),
        dtype=torch.float32
    ).view(-1, 1)

    # 5. Process each edge dataframe (for each edge type)
    edge_type_stats = {}
    for edge_type, edges in edge_dataframes.items():
        # 5a. Validate edges: ensure source and target are valid substations
        valid_mask = (
            edges['source'].isin(valid_subs) & 
            edges['target'].isin(valid_subs)
        )
        filtered_edges = edges[valid_mask].copy()
        
        # 5b. Deduplicate edges based on source and target
        filtered_edges = filtered_edges.drop_duplicates(subset=['source', 'target'])
        
        # 5c. Ensure edge attribute columns are numeric and safe
        edge_attr = filtered_edges.drop(columns=['source', 'target'])\
            .apply(pd.to_numeric, errors='coerce')\
            .fillna(0)\
            .astype(np.float32)
            
        # 5d. Map source and target substations to indices
        source_idx = filtered_edges['source'].map(
            {sub: idx for idx, sub in enumerate(valid_subs)}
        ).values
        target_idx = filtered_edges['target'].map(
            {sub: idx for idx, sub in enumerate(valid_subs)}
        ).values
        
        # 5e. Create edge index and edge attribute tensors
        data['substation', edge_type, 'substation'].edge_index = torch.tensor(
            [source_idx, target_idx], dtype=torch.long
        )
        data['substation', edge_type, 'substation'].edge_attr = torch.tensor(
            edge_attr.values, dtype=torch.float32
        )
        
        edge_type_stats[edge_type] = len(filtered_edges)

    # 6. Final graph validation output
    print("\nGraph Integrity Check:")
    print(f"Nodes: {len(valid_subs)}")
    print(f"Features: {data['substation'].x.shape}")
    print(f"Targets: {data['substation'].y.shape}")
    print("Edge Counts:")
    for et, count in edge_type_stats.items():
        print(f"- {et}: {count} edges")
        
    # 7. Save the graph with versioning for reproducibility
    save_path = f"power_grid_graph_{pd.Timestamp.now().strftime('%Y%m%d')}.pt"
    torch.save(data, save_path)
    print(f"\nGraph saved to {save_path}")
    
    return data

# Enhanced execution with validation:

hetero_graph = build_heterogeneous_graph(
    node_features=node_features_full,
    target_variable=full_targets,  # targets computed with the 180-day window logic
    edge_dataframes={
        'spatial': spatial_edges,
        'temporal': temporal_edges,
        'causal': causal_edges
    }
)



Reindexing target variable to match node features and filling missing targets with 0.

Graph Integrity Check:
Nodes: 380
Features: torch.Size([380, 4])
Targets: torch.Size([380, 1])
Edge Counts:
- spatial: 6162 edges
- temporal: 59908 edges
- causal: 3839 edges

Graph saved to power_grid_graph_20250222.pt


In [11]:
# Cell 09: Splitting 

# Total number of nodes in the 'substation' node type
num_nodes = hetero_graph['substation'].x.size(0)

# Create a randomized permutation of node indices
indices = torch.randperm(num_nodes)

# Define split sizes (e.g., 70% train, 15% validation, 15% test)
train_size = int(0.7 * num_nodes)
val_size = int(0.15 * num_nodes)
test_size = num_nodes - train_size - val_size

# Get the indices for each split
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

# Create boolean masks for each split
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_indices] = True
val_mask[val_indices] = True
test_mask[test_indices] = True

# Assign the masks to the hetero graph's 'substation' node data
hetero_graph['substation'].train_mask = train_mask
hetero_graph['substation'].val_mask = val_mask
hetero_graph['substation'].test_mask = test_mask

print("Masks assigned:")
print(f"Train nodes: {train_mask.sum().item()}, Validation nodes: {val_mask.sum().item()}, Test nodes: {test_mask.sum().item()}")


Masks assigned:
Train nodes: 266, Validation nodes: 57, Test nodes: 57


In [12]:
# Cell 10: Leak-Safe Weighted Convolution
class WeightedGCNConv(nn.Module):
    def __init__(self, in_channels, out_channels, edge_attr_dim):
        super().__init__()
        # Use GCNConv without self-loops (they can be added externally if needed)
        self.gcn = GCNConv(in_channels, out_channels, add_self_loops=False)
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_attr_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()  # Constrain weights to [0,1]
        )
        self._init_weights()

    def _init_weights(self):
        for layer in self.edge_mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.constant_(layer.bias, 0.1)

    def forward(self, x, edge_index, edge_attr):
        """
        Forward pass for the weighted GCN convolution.

        Parameters:
          x: Node feature tensor (e.g. data['substation'].x from our hetero graph)
          edge_index: Edge indices tensor for a chosen edge type (e.g. data['substation', 'spatial', 'substation'].edge_index)
          edge_attr: Edge attribute tensor corresponding to the edge_index

        Returns:
          out: The updated node features after applying the weighted GCNConv.
        """
        # If there are no edges, return zeros for this layer's contribution.
        if edge_index.size(1) == 0:
            return torch.zeros(x.size(0), self.gcn.out_channels, device=x.device)

        # Ensure edge_attr is at least 2D.
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
        
        # Compute edge weights using the edge MLP.
        weights = self.edge_mlp(edge_attr).squeeze()
        # Add a small epsilon to prevent division by zero in GCNConv.
        weights = weights + 1e-8

        # (Optional) Debug: Uncomment to print edge weight statistics.
        # print("Edge weights - min:", weights.min().item(), 
        #       "max:", weights.max().item(), 
        #       "mean:", weights.mean().item())

        # Pass the computed weights to the GCNConv layer via the edge_weight parameter.
        out = self.gcn(x, edge_index, edge_weight=weights)
        return out


In [13]:
# Cell 11: Enhanced Heterogeneous GNN (Revised with Device-Moved Edge Masks)

class PowerGridGNN(nn.Module):
    def __init__(self, in_channels, hidden_dim, edge_dims, num_layers=3):
        super().__init__()
        self.edge_types = ['spatial', 'temporal', 'causal']
        self.layers = nn.ModuleList()
        current_dim = in_channels
        
        # Layer-wise initialization with dropout
        for _ in range(num_layers):
            layer = nn.ModuleDict({
                et: WeightedGCNConv(current_dim, hidden_dim, edge_dims[et])
                for et in self.edge_types
            })
            self.layers.append(layer)
            current_dim = hidden_dim
            
        # Final prediction with regularization
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Kaiming initialization for all linear layers
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, data):
        x = data['substation'].x
        edge_masks = self._get_edge_masks(data)  # Create edge masks
        
        for layer in self.layers:
            messages = []
            for et in self.edge_types:
                edge_info = data['substation', et, 'substation']
                idx = edge_info.edge_index
                attr = edge_info.edge_attr
                # Move the mask to the same device as x
                mask = edge_masks[et].to(x.device)
                if mask is not None:
                    idx = idx[:, mask]
                    attr = attr[mask]
                messages.append(layer[et](x, idx, attr))
            
            x = sum(messages)
            x = F.relu(x)
            x = F.dropout(x, p=0.3, training=self.training)
            
        return self.head(x)

    def _get_edge_masks(self, data):
        """Create edge masks to isolate training nodes."""
        masks = {}
        train_mask = data['substation'].train_mask.bool()  # Critical: Use stored boolean mask
        for et in self.edge_types:
            edge_info = data['substation', et, 'substation']
            # Mask edges where BOTH nodes are in training
            source_train = train_mask[edge_info.edge_index[0]]
            target_train = train_mask[edge_info.edge_index[1]]
            masks[et] = source_train & target_train  # Both must be True
        return masks


In [16]:
# Cell 12: Training and Evaluation on Full Graph (With Strict Masking)

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Assume train_mask, val_mask, test_mask are already created and stored in variables ---
# Store masks in the graph (re-assigning here for clarity)
hetero_graph['substation'].train_mask = train_mask
hetero_graph['substation'].val_mask = val_mask
hetero_graph['substation'].test_mask = test_mask

# --- Move Entire Graph to Device ---
hetero_graph = hetero_graph.to(device)

# --- Define Hyperparameters ---
in_channels = node_features_full.shape[1]
hidden_channels = 100
num_layers = 2
num_epochs = 100

#---- Edge Dictionary 

edge_attr_dims = {
    'spatial': 3,   # e.g., distance, shared_cities, shared_feeders
    'temporal': 2,  # e.g., min_duration, event_count
    'causal': 6     # e.g., cause (converted to 0), mean_lag_hrs, distance_m, event_count, norm_lag, norm_dist
}


# --- Model Initialization ---
model = PowerGridGNN(
    in_channels=in_channels,
    hidden_dim=hidden_channels,
    edge_dims=edge_attr_dims,  # Ensure edge_attr_dims is defined
    num_layers=num_layers
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()

# --- Training Loop ---
best_val_loss = float('inf')
for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Clone graph and mask non-training nodes
    train_data = hetero_graph.clone()
    train_data['substation'].x[~train_mask] = 0  # Zero out features of non-training nodes
    train_data['substation'].y[~train_mask] = float('nan')  # Hide labels of non-training nodes
    
    out = model(train_data)
    loss = criterion(out[train_mask].squeeze(), train_data['substation'].y[train_mask].squeeze())
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    # --- Validation ---
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_data = hetero_graph.clone()
            # Set non-validation nodes' labels to NaN so that loss is computed only on validation nodes
            val_data['substation'].y[~val_mask] = float('nan')
            out_val = model(val_data)
            val_loss = criterion(out_val[val_mask].squeeze(), val_data['substation'].y[val_mask].squeeze())
            
        print(f"Epoch {epoch} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

# --- Final Test ---
model.eval()
with torch.no_grad():
    test_data = hetero_graph.clone()
    test_data['substation'].y[~test_mask] = float('nan')
    out_test = model(test_data)
    predictions = (torch.sigmoid(out_test) > 0.5).float()
    correct = (predictions[test_mask].squeeze() == test_data['substation'].y[test_mask].squeeze()).sum().item()
    acc = correct / test_mask.sum().item()
    print(f"Test Accuracy: {acc:.4f}")


Epoch 10 | Train Loss: 1.3648 | Val Loss: 0.6181
Epoch 20 | Train Loss: 0.8265 | Val Loss: 0.5777
Epoch 30 | Train Loss: 0.7413 | Val Loss: 0.5680
Epoch 40 | Train Loss: 1.0560 | Val Loss: 0.5402
Epoch 50 | Train Loss: 0.6618 | Val Loss: 0.5391
Epoch 60 | Train Loss: 0.5984 | Val Loss: 0.5418
Epoch 70 | Train Loss: 0.5419 | Val Loss: 0.5460
Epoch 80 | Train Loss: 0.5003 | Val Loss: 0.5440
Epoch 90 | Train Loss: 0.5706 | Val Loss: 0.5444
Epoch 100 | Train Loss: 0.5253 | Val Loss: 0.5384
Test Accuracy: 0.7544


In [18]:
import numpy as np
from sklearn.model_selection import KFold

# Define hyperparameter grid
learning_rates = [0.001, 0.0005]
hidden_dims = [100, 150]
num_layers_options = [2, 3]
num_epochs_cv = 100  # You may want to reduce this during search to speed up experiments

num_nodes = hetero_graph['substation'].x.size(0)
all_indices = np.arange(num_nodes)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for lr in learning_rates:
    for hidden_dim in hidden_dims:
        for num_layers in num_layers_options:
            fold_val_losses = []
            fold_test_accuracies = []
            print(f"\nTesting hyperparameters: lr={lr}, hidden_dim={hidden_dim}, num_layers={num_layers}")
            for fold, (train_val_idx, test_idx) in enumerate(kf.split(all_indices)):
                # Further split train_val_idx into training and validation (e.g., 85% train, 15% val)
                train_val_idx = np.array(train_val_idx)
                np.random.shuffle(train_val_idx)
                val_size = int(0.15 * len(train_val_idx))
                val_idx = train_val_idx[:val_size]
                train_idx = train_val_idx[val_size:]
                
                # Create boolean masks for this fold
                train_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
                val_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
                test_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
                train_mask_fold[train_idx] = True
                val_mask_fold[val_idx] = True
                test_mask_fold[test_idx] = True
                
                # Clone the full hetero graph for this fold and assign masks
                fold_graph = hetero_graph.clone()
                fold_graph['substation'].train_mask = train_mask_fold
                fold_graph['substation'].val_mask = val_mask_fold
                fold_graph['substation'].test_mask = test_mask_fold
                
                # Initialize model for this fold with current hyperparameters
                model_fold = PowerGridGNN(
                    in_channels=in_channels,
                    hidden_dim=hidden_dim,
                    edge_dims=edge_attr_dims,
                    num_layers=num_layers
                ).to(device)
                optimizer_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)
                criterion_fold = torch.nn.BCEWithLogitsLoss()
                
                # Training loop for the fold
                for epoch in range(1, num_epochs_cv + 1):
                    model_fold.train()
                    optimizer_fold.zero_grad()
                    
                    # Mask non-training nodes
                    train_data = fold_graph.clone()
                    train_data['substation'].x[~train_mask_fold] = 0
                    train_data['substation'].y[~train_mask_fold] = float('nan')
                    
                    out_fold = model_fold(train_data)
                    loss_fold = criterion_fold(
                        out_fold[train_mask_fold].squeeze(), 
                        train_data['substation'].y[train_mask_fold].squeeze()
                    )
                    loss_fold.backward()
                    torch.nn.utils.clip_grad_norm_(model_fold.parameters(), max_norm=1.0)
                    optimizer_fold.step()
                
                # Evaluate on validation set
                model_fold.eval()
                with torch.no_grad():
                    val_data = fold_graph.clone()
                    val_data['substation'].y[~val_mask_fold] = float('nan')
                    out_val_fold = model_fold(val_data)
                    val_loss_fold = criterion_fold(
                        out_val_fold[val_mask_fold].squeeze(), 
                        val_data['substation'].y[val_mask_fold].squeeze()
                    )
                    
                    # Evaluate on test set for reporting purposes
                    test_data = fold_graph.clone()
                    test_data['substation'].y[~test_mask_fold] = float('nan')
                    out_test_fold = model_fold(test_data)
                    predictions_fold = (torch.sigmoid(out_test_fold) > 0.5).float()
                    correct_fold = (predictions_fold[test_mask_fold].squeeze() == 
                                    test_data['substation'].y[test_mask_fold].squeeze()).sum().item()
                    test_acc_fold = correct_fold / test_mask_fold.sum().item()
                
                fold_val_losses.append(val_loss_fold.item())
                fold_test_accuracies.append(test_acc_fold)
                print(f"  Fold {fold+1} | Val Loss: {val_loss_fold.item():.4f} | Test Acc: {test_acc_fold:.4f}")
            
            avg_val_loss = np.mean(fold_val_losses)
            avg_test_acc = np.mean(fold_test_accuracies)
            results.append({
                'lr': lr,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'avg_val_loss': avg_val_loss,
                'avg_test_acc': avg_test_acc
            })
            print(f"--> Avg Val Loss: {avg_val_loss:.4f} | Avg Test Acc: {avg_test_acc:.4f}")

# Identify best hyperparameter configuration (here, minimizing validation loss)
best_config = min(results, key=lambda x: x['avg_val_loss'])
print("\nBest hyperparameter configuration based on validation loss:")
print(best_config)



Testing hyperparameters: lr=0.001, hidden_dim=100, num_layers=2
  Fold 1 | Val Loss: 0.6985 | Test Acc: 0.1842
  Fold 2 | Val Loss: 0.4728 | Test Acc: 0.7763
  Fold 3 | Val Loss: 1.7954 | Test Acc: 0.2500
  Fold 4 | Val Loss: 1.0290 | Test Acc: 0.1974
  Fold 5 | Val Loss: 0.5119 | Test Acc: 0.7895
--> Avg Val Loss: 0.9015 | Avg Test Acc: 0.4395

Testing hyperparameters: lr=0.001, hidden_dim=100, num_layers=3
  Fold 1 | Val Loss: 0.4894 | Test Acc: 0.8158
  Fold 2 | Val Loss: 0.5299 | Test Acc: 0.7763
  Fold 3 | Val Loss: 0.5912 | Test Acc: 0.7500
  Fold 4 | Val Loss: 0.6012 | Test Acc: 0.8026
  Fold 5 | Val Loss: 0.5677 | Test Acc: 0.7895
--> Avg Val Loss: 0.5559 | Avg Test Acc: 0.7868

Testing hyperparameters: lr=0.001, hidden_dim=150, num_layers=2
  Fold 1 | Val Loss: 0.6174 | Test Acc: 0.8158
  Fold 2 | Val Loss: 0.5229 | Test Acc: 0.7763
  Fold 3 | Val Loss: 2.3307 | Test Acc: 0.2500
  Fold 4 | Val Loss: 2.2391 | Test Acc: 0.1974
  Fold 5 | Val Loss: 0.8052 | Test Acc: 0.2105
--> 

In [19]:
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Best configuration from hyperparameter search:
best_lr = 0.001
best_hidden_dim = 100
best_num_layers = 3
num_epochs = 100
patience = 10  # Early stopping patience

# Extract labels from the hetero graph (assume shape is [N, 1])
labels = hetero_graph['substation'].y.squeeze().cpu().numpy().astype(int)
num_nodes = len(labels)

# Use StratifiedKFold to preserve the label distribution across folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_val_losses = []
fold_test_accs = []
fold_idx = 1

for train_val_idx, test_idx in skf.split(np.arange(num_nodes), labels):
    # Further split train_val into training and validation (e.g., 85% train, 15% val)
    train_val_idx = np.array(train_val_idx)
    np.random.shuffle(train_val_idx)
    val_size = int(0.15 * len(train_val_idx))
    val_idx = train_val_idx[:val_size]
    train_idx = train_val_idx[val_size:]
    
    # Create boolean masks for this fold
    train_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask_fold = torch.zeros(num_nodes, dtype=torch.bool)
    train_mask_fold[train_idx] = True
    val_mask_fold[val_idx] = True
    test_mask_fold[test_idx] = True
    
    # Clone the hetero graph for this fold and assign masks
    fold_graph = hetero_graph.clone()
    fold_graph['substation'].train_mask = train_mask_fold
    fold_graph['substation'].val_mask = val_mask_fold
    fold_graph['substation'].test_mask = test_mask_fold
    
    # Initialize model, optimizer, scheduler, and criterion for this fold
    model_fold = PowerGridGNN(
        in_channels=in_channels,
        hidden_dim=best_hidden_dim,
        edge_dims=edge_attr_dims,
        num_layers=best_num_layers
    ).to(device)
    
    optimizer_fold = torch.optim.Adam(model_fold.parameters(), lr=best_lr)
    scheduler_fold = ReduceLROnPlateau(optimizer_fold, mode='min', factor=0.5, patience=5, verbose=True)
    criterion_fold = torch.nn.BCEWithLogitsLoss()
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_epoch = 0
    best_model_state = None
    
    # Training loop with early stopping
    for epoch in range(1, num_epochs + 1):
        model_fold.train()
        optimizer_fold.zero_grad()
        
        # Create a clone and mask non-training nodes:
        train_data = fold_graph.clone()
        train_data['substation'].x[~train_mask_fold] = 0
        train_data['substation'].y[~train_mask_fold] = float('nan')
        
        out = model_fold(train_data)
        loss = criterion_fold(out[train_mask_fold].squeeze(), train_data['substation'].y[train_mask_fold].squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_fold.parameters(), max_norm=1.0)
        optimizer_fold.step()
        
        # Validation evaluation
        model_fold.eval()
        with torch.no_grad():
            val_data = fold_graph.clone()
            # Mask non-validation nodes so that loss is computed only on val nodes
            val_data['substation'].y[~val_mask_fold] = float('nan')
            out_val = model_fold(val_data)
            val_loss = criterion_fold(out_val[val_mask_fold].squeeze(), val_data['substation'].y[val_mask_fold].squeeze())
        
        # Step the scheduler using validation loss
        scheduler_fold.step(val_loss)
        
        # Check for early stopping
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_epoch = epoch
            epochs_no_improve = 0
            best_model_state = model_fold.state_dict()
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f"Fold {fold_idx} Early stopping at epoch {epoch}")
            break
        
        if epoch % 10 == 0:
            print(f"Fold {fold_idx} | Epoch {epoch} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    
    # Load best model for testing
    model_fold.load_state_dict(best_model_state)
    model_fold.eval()
    with torch.no_grad():
        test_data = fold_graph.clone()
        test_data['substation'].y[~test_mask_fold] = float('nan')
        out_test = model_fold(test_data)
        predictions = (torch.sigmoid(out_test) > 0.5).float()
        correct = (predictions[test_mask_fold].squeeze() == test_data['substation'].y[test_mask_fold].squeeze()).sum().item()
        test_acc = correct / test_mask_fold.sum().item()
    
    print(f"Fold {fold_idx} | Best Epoch: {best_epoch} | Test Accuracy: {test_acc:.4f}")
    fold_val_losses.append(best_val_loss)
    fold_test_accs.append(test_acc)
    fold_idx += 1

print(f"\nAverage Validation Loss: {np.mean(fold_val_losses):.4f}")
print(f"Average Test Accuracy: {np.mean(fold_test_accs):.4f}")


Fold 1 | Epoch 10 | Train Loss: 0.5907 | Val Loss: 0.5598
Fold 1 | Epoch 20 | Train Loss: 0.6635 | Val Loss: 0.5339
Fold 1 | Epoch 30 | Train Loss: 0.5875 | Val Loss: 0.4685
Fold 1 | Epoch 40 | Train Loss: 0.8445 | Val Loss: 0.4533
Fold 1 | Epoch 50 | Train Loss: 1.0615 | Val Loss: 0.4381
Fold 1 Early stopping at epoch 60
Fold 1 | Best Epoch: 50 | Test Accuracy: 0.7895
Fold 2 | Epoch 10 | Train Loss: 0.5164 | Val Loss: 0.6056
Fold 2 Early stopping at epoch 15
Fold 2 | Best Epoch: 5 | Test Accuracy: 0.7895
Fold 3 | Epoch 10 | Train Loss: 1.8123 | Val Loss: 0.5081
Fold 3 Early stopping at epoch 16
Fold 3 | Best Epoch: 6 | Test Accuracy: 0.7895
Fold 4 | Epoch 10 | Train Loss: 0.8510 | Val Loss: 0.6345
Fold 4 Early stopping at epoch 17
Fold 4 | Best Epoch: 7 | Test Accuracy: 0.7895
Fold 5 | Epoch 10 | Train Loss: 1.3889 | Val Loss: 0.6426
Fold 5 Early stopping at epoch 13
Fold 5 | Best Epoch: 3 | Test Accuracy: 0.7763

Average Validation Loss: 0.5360
Average Test Accuracy: 0.7868


In [23]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Best hyperparameters from hyperparameter search
best_lr = 0.001
best_hidden_dim = 100
best_num_layers = 3
num_epochs = 200         # Increase total epochs
patience = 20            # Increase early stopping patience

# Ensure the full hetero_graph has the masks assigned and move it to device
hetero_graph['substation'].train_mask = train_mask
hetero_graph['substation'].val_mask = val_mask
hetero_graph['substation'].test_mask = test_mask
hetero_graph = hetero_graph.to(device)

# Define input channels based on your node features
in_channels = node_features_full.shape[1]

# Model Initialization with best hyperparameters
model = PowerGridGNN(
    in_channels=in_channels,
    hidden_dim=best_hidden_dim,
    edge_dims=edge_attr_dims,  # e.g., {'spatial': 3, 'temporal': 2, 'causal': 6}
    num_layers=best_num_layers
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
# Adjust scheduler: increase patience and use a factor closer to 1 (less aggressive reduction)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=10, verbose=True)
criterion = torch.nn.BCEWithLogitsLoss()

best_val_loss = float('inf')
epochs_no_improve = 0
best_epoch = 0
best_model_state = copy.deepcopy(model.state_dict())

print("Starting full-data training with increased epochs and patience...")

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Clone graph and mask non-training nodes (features zeroed, labels hidden)
    train_data = hetero_graph.clone()
    train_data['substation'].x[~train_mask] = 0
    train_data['substation'].y[~train_mask] = float('nan')
    
    out = model(train_data)
    loss = criterion(out[train_mask].squeeze(), train_data['substation'].y[train_mask].squeeze())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Evaluate on validation set
    model.eval()
    with torch.no_grad():
        val_data = hetero_graph.clone()
        val_data['substation'].y[~val_mask] = float('nan')
        out_val = model(val_data)
        val_loss = criterion(out_val[val_mask].squeeze(), val_data['substation'].y[val_mask].squeeze())
    
    # Step the scheduler based on validation loss
    scheduler.step(val_loss)
    
    # Check for improvement for early stopping
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_epoch = epoch
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# Load best model weights based on validation loss
model.load_state_dict(best_model_state)
print(f"Best validation loss {best_val_loss:.4f} achieved at epoch {best_epoch}")

# Evaluate on the test set
model.eval()
with torch.no_grad():
    test_data = hetero_graph.clone()
    test_data['substation'].y[~test_mask] = float('nan')
    out_test = model(test_data)
    predictions = (torch.sigmoid(out_test) > 0.5).float()
    correct = (predictions[test_mask].squeeze() == test_data['substation'].y[test_mask].squeeze()).sum().item()
    test_acc = correct / test_mask.sum().item()

print(f"Final Test Accuracy: {test_acc:.4f}")


Starting full-data training with increased epochs and patience...
Epoch 10 | Train Loss: 2.8924 | Val Loss: 0.8789
Epoch 20 | Train Loss: 2.1060 | Val Loss: 0.7373
Epoch 30 | Train Loss: 1.7720 | Val Loss: 0.6483
Epoch 40 | Train Loss: 1.6902 | Val Loss: 0.5918
Epoch 50 | Train Loss: 1.3286 | Val Loss: 0.5672
Epoch 60 | Train Loss: 1.5469 | Val Loss: 0.5585
Epoch 70 | Train Loss: 1.2955 | Val Loss: 0.5535
Epoch 80 | Train Loss: 0.8687 | Val Loss: 0.5434
Epoch 90 | Train Loss: 0.5719 | Val Loss: 0.5399
Epoch 100 | Train Loss: 0.4956 | Val Loss: 0.5377
Early stopping at epoch 104
Best validation loss 0.5369 achieved at epoch 84
Final Test Accuracy: 0.7544


In [ ]:
def analyze_edge_weights(model, hetero_graph, relation, layer_idx=0):
    """
    Analyzes statistical properties of learned edge weights for a given relation.
    
    Parameters:
      - model: Your trained GNN model.
      - hetero_graph: Your HeteroData object.
      - relation: One of 'spatial', 'temporal', or 'causal'.
      - layer_idx: Which layer's edge_mlp to use (default is first layer).
    """
    # Ensure model is on the correct device
    device = next(model.parameters()).device
    model.to(device)
    
    # Extract edge attributes and move them to the correct device
    edge_attr = hetero_graph['substation', relation, 'substation'].edge_attr.to(device)

    # Get the edge_mlp for the specified relation and layer
    edge_mlp = model.layers[layer_idx][relation].edge_mlp
    with torch.no_grad():
        # Compute edge weights using the edge_mlp and then move to CPU for analysis
        weights = edge_mlp(edge_attr).squeeze(-1).cpu().numpy()
    
    # Compute statistics
    mean_weight = np.mean(weights)
    median_weight = np.median(weights)
    std_dev = np.std(weights)
    min_weight = np.min(weights)
    max_weight = np.max(weights)
    zero_count = np.sum(weights == 0)
    
    print(f"\n🔹 Edge Weight Analysis for '{relation}' Relation 🔹")
    print(f"Mean Weight: {mean_weight:.4f}")
    print(f"Median Weight: {median_weight:.4f}")
    print(f"Standard Deviation: {std_dev:.4f}")
    print(f"Min Weight: {min_weight:.4f}")
    print(f"Max Weight: {max_weight:.4f}")
    print(f"Zero Weights Count: {zero_count}/{len(weights)} ({(zero_count / len(weights)) * 100:.2f}%)\n")

    # Plot histogram using Matplotlib
    plt.figure(figsize=(8, 5))
    plt.hist(weights, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    plt.title(f"Edge Weight Distribution for '{relation}' Relation")
    plt.xlabel("Edge Weight")
    plt.ylabel("Frequency")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

# --- Load the Best Model and Set It to Evaluation Mode ---
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

# --- Analyze Edge Weights for Each Relation ---
for relation in ['spatial', 'temporal', 'causal']:
    analyze_edge_weights(model, hetero_graph, relation)


In [ ]:
data_hp = hetero_graph.clone().to(device)


In [ ]:
# Hyperparameter Tuning (Simple Grid Search Outline)

# Define a grid of hyperparameters to try
learning_rates = [0.005, 0.01, 0.02]
hidden_dims = [32, 64, 128]
num_layers_list = [1, 2]

best_val_acc = 0.0
best_params = {}
results = []

for lr in learning_rates:
    for hidden in hidden_dims:
        for num_layers in num_layers_list:
            print(f"\nTraining with lr={lr}, hidden_dim={hidden}, num_layers={num_layers}")
            # Re-create model and optimizer with current hyperparameters
            model_hp = PowerGridGNN(
                in_channels=in_channels,
                hidden_dim=hidden,
                edge_dims=edge_attr_dims,
                num_layers=num_layers
            ).to(device)
            optimizer_hp = torch.optim.Adam(model_hp.parameters(), lr=lr)
            
            # IMPORTANT: Clone and move hetero_graph to device for this run
            data_hp = hetero_graph.clone().to(device)
            
            # Use the existing train/val split from data_hp
            num_epochs_hp = 50
            best_val_loss_hp = float('inf')
            for epoch in range(1, num_epochs_hp + 1):
                model_hp.train()
                optimizer_hp.zero_grad()
                out_hp = model_hp(data_hp)  # data_hp is on device
                train_loss = criterion(out_hp[train_mask].squeeze(), data_hp['substation'].y[train_mask].squeeze())
                train_loss.backward()
                optimizer_hp.step()
                
                model_hp.eval()
                with torch.no_grad():
                    out_val = model_hp(data_hp)
                    val_loss = criterion(out_val[val_mask].squeeze(), data_hp['substation'].y[val_mask].squeeze())
                    
                # Save best validation loss for this hyperparameter setting
                if val_loss < best_val_loss_hp:
                    best_val_loss_hp = val_loss
            
            # After training, evaluate validation accuracy
            model_hp.eval()
            with torch.no_grad():
                out_val = model_hp(data_hp)
                preds_val = (torch.sigmoid(out_val) > 0.5).float()
                correct_val = (preds_val[val_mask].squeeze() == data_hp['substation'].y[val_mask].squeeze()).sum().item()
                val_acc = correct_val / val_mask.sum().item()
            
            print(f"Validation Accuracy: {val_acc:.4f}")
            results.append({'lr': lr, 'hidden': hidden, 'num_layers': num_layers, 'val_acc': val_acc})
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_params = {'lr': lr, 'hidden': hidden, 'num_layers': num_layers}

print("\nBest Hyperparameters:", best_params)
